In [29]:
import json
from snowflake import connector
import huggingface_hub
import os

In [30]:
with open('../snowflake_credentials.json') as f:
    snowflake_credentials = json.load(f)

In [31]:
# connecting to snowflake
conn = connector.connect(
    user=snowflake_credentials['username'],
    password=snowflake_credentials['password'],
    account=snowflake_credentials['account'],
    role=snowflake_credentials['role'],
    warehouse=snowflake_credentials['warehouse']
)

In [32]:
cursor = conn.cursor()

In [33]:
# show all databases

result = cursor.execute("SHOW DATABASES")
databases = [row[1] for row in result]

In [34]:
def get_tables(database):
    all_tables_info = []
    try:
        # Sử dụng database
        cursor.execute(f'USE DATABASE "{database}"')
        
        # SHOW TABLES trả về rất nhiều cột:
        # index 1: tên table, index 2: tên database, index 3: tên schema
        cursor.execute("SHOW TABLES")
        result = cursor.fetchall()
        
        for row in result:
            schema_name = row[3]  # Vị trí schema trong SHOW TABLES
            table_name = row[1]   # Vị trí table name
            all_tables_info.append((schema_name, table_name))
            
        return all_tables_info
    
    except Exception as e:
        # Nếu gặp lỗi "Shared database is no longer available" hoặc "Not authorized"
        # hàm sẽ bắt lỗi ở đây và skip database này
        print(f"⚠️ Bỏ qua Database '{database}': {e}")
        return []

In [35]:
def get_columns(database, schema, table):
    try:
        # Sử dụng đường dẫn tuyệt đối để tránh lỗi "not exist"
        query = f'SHOW COLUMNS IN TABLE "{database}"."{schema}"."{table}"'
        cursor.execute(query)
        return cursor.fetchall()
    except Exception as e:
        print(f"Lỗi tại {database}.{table}: {e}")
        return []

In [36]:
columns_data = []

for db in databases:
    tables_in_db = get_tables(db) # Trả về danh sách các tuple (schema, table)
    
    for schema, table in tables_in_db:
        try:
            # Truy vấn trực tiếp bằng đường dẫn: "DATABASE"."SCHEMA"."TABLE"
            query = f'SHOW COLUMNS IN TABLE "{db}"."{schema}"."{table}"'
            cursor.execute(query)
            
            rows = cursor.fetchall()
            for col in rows:
                # Lưu thông tin (Tên DB, Tên Schema, Tên Table, Tên Column)
                columns_data.append({
                    'database': db,
                    'schema': schema,
                    'table': table,
                    'column': col[2] # Index 2 thường là tên cột trong SHOW COLUMNS
                })
        except Exception as e:
            # Nếu bảng tồn tại nhưng bạn không có quyền xem Column (Not authorized)
            # nó sẽ lọt vào đây và chương trình vẫn tiếp tục chạy bảng khác.
            print(f"🚫 Skip table {db}.{schema}.{table} - Lỗi: {e}")

⚠️ Bỏ qua Database 'AMAZON_VENDOR_ANALYTICS__SAMPLE_DATASET': 003030 (02000): SQL compilation error:
Shared database is no longer available for use. It will need to be re-created if and when the publisher makes it available again.
⚠️ Bỏ qua Database 'NETHERLANDS_OPEN_MAP_DATA': 003030 (02000): SQL compilation error:
Shared database is no longer available for use. It will need to be re-created if and when the publisher makes it available again.


In [37]:
print(f"Tổng số cột đã thu thập: {len(columns_data)}")

Tổng số cột đã thu thập: 199085


In [38]:
with open('../preprocessed_data/snowflake_columns.json', 'w') as f:
    json.dump(columns_data, f, indent=4)